# Robust LightGCN with Features — Multi-Quarter Sweep

Trains `WeightedLightGCN` on every available quarter in `changed_holdings.parquet` and saves:
- **Per-quarter metrics CSV**: AUC, P/R/F1, Top-K hit rate / NDCG
- **Per-stock rank parquet**: for each `(year, quarter)`, every CUSIP gets a `mean_score` (mean sigmoid affinity over all funds) and an integer `rank` (1 = highest predicted attractiveness)

Both files are appended incrementally so the sweep is **resumable** — re-run to pick up where it left off.

## 1. Configuration

In [ ]:
EDGES_COLUMN = "change_in_weight"  # or "change_in_adjusted_weight"
DATA_DIR = "../../Data/parquet_for_cluster"
RESULTS_DIR = "./results"   # outputs land here

# Hyperparameters (mirror the dev notebook)
EMBEDDING_DIM = 128
NUM_LAYERS = 3
EPOCHS = 50
LEARNING_RATE = 0.001
EARLY_STOPPING_PATIENCE = 10

TRAIN_RATIO = 0.8
VAL_RATIO   = 0.1
TEST_RATIO  = 0.1
RANDOM_SEED = 42
TOP_K_VALUES = [5, 10, 20, 50]

# Optional: restrict the sweep. None = all available quarters.
# Example: QUARTERS_OVERRIDE = [(2021, 4), (2022, 1)]
QUARTERS_OVERRIDE = None

## 2. Imports

In [ ]:
import os
import time
import random
import warnings
import traceback
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv

from sklearn.metrics import (
    roc_auc_score, precision_score, recall_score, f1_score,
    precision_recall_curve, average_precision_score,
)
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

np.random.seed(RANDOM_SEED); random.seed(RANDOM_SEED); torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch: {torch.__version__}  device: {DEVICE}")

## 3. Load Korkevados parquets (once, into globals)

In [ ]:
DATA_PATH = Path(DATA_DIR).expanduser().resolve()
if not DATA_PATH.is_dir():
    raise FileNotFoundError(f"DATA_DIR not found: {DATA_PATH}")

RESULTS_PATH = Path(RESULTS_DIR).expanduser().resolve()
RESULTS_PATH.mkdir(parents=True, exist_ok=True)
print(f"DATA_PATH    : {DATA_PATH}")
print(f"RESULTS_PATH : {RESULTS_PATH}")

def _read_parquet(name):
    p = DATA_PATH / f"{name}.parquet"
    df = pd.read_parquet(p) if p.exists() else pd.DataFrame()
    print(f"  loaded {name:30s} {len(df):>10,} rows")
    return df

CHANGED_HOLDINGS = _read_parquet("changed_holdings")
STOCKS_RETURN    = _read_parquet("stocks_return")
CIK_AUM          = _read_parquet("cik_aum")
NORM_HOLDINGS    = _read_parquet("normalized_holdings")
CUSIP_FIN        = _read_parquet("cusip_financial_data")

## 4. Loaders, graph builder, model (copied from dev notebook)

In [ ]:
STOCK_FEATURE_COLS = [
    "diluted_eps", "roe", "ev_ebitda", "pe_ratio", "price_to_sales",
    "price_to_book", "debt_to_equity", "dividend_yield", "fcf_per_share",
    "log_return",
]

def next_year_quarter(y, q): return (y + 1, 1) if q == 4 else (y, q + 1)
def prev_year_quarter(y, q): return (y - 1, 4) if q == 1 else (y, q - 1)


def load_edges(year, quarter, col):
    df = CHANGED_HOLDINGS
    if df.empty or col not in df.columns:
        return pd.DataFrame(columns=["cik", "cusip", "w"])
    mask = (df["year"] == year) & (df["quarter"] == quarter) & df[col].notna()
    return (df.loc[mask, ["cik", "cusip", col]]
              .rename(columns={col: "w"}).reset_index(drop=True))


def load_returns(year, quarter):
    df = STOCKS_RETURN
    if df.empty:
        return pd.DataFrame(columns=["cusip", "log_return"])
    mask = (df["year"] == year) & (df["quarter"] == quarter) & df["log_return"].notna()
    return df.loc[mask, ["cusip", "log_return"]].reset_index(drop=True)


def load_aum(year, quarter):
    df = CIK_AUM
    if df.empty:
        return pd.DataFrame(columns=["cik", "aum"])
    mask = (df["year"] == year) & (df["quarter"] == quarter) & (df["total"] > 0)
    return (df.loc[mask, ["cik", "total"]]
              .rename(columns={"total": "aum"}).reset_index(drop=True))


def load_stock_features(year, quarter):
    fin = CUSIP_FIN
    if fin.empty:
        fin = pd.DataFrame(columns=["cusip"] + STOCK_FEATURE_COLS)
    else:
        fin = fin.loc[(fin["year"] == year) & (fin["quarter"] == quarter)].copy()
    rets = load_returns(year, quarter)
    df = fin.merge(rets, on="cusip", how="outer")
    for c in STOCK_FEATURE_COLS:
        if c not in df.columns:
            df[c] = 0.0
    return df[["cusip"] + STOCK_FEATURE_COLS]


def investor_profitability(year, quarter):
    ny, nq = next_year_quarter(year, quarter)
    h = NORM_HOLDINGS
    if h.empty:
        return pd.Series(dtype=float, name="profitability")
    h = h.loc[(h["year"] == year) & (h["quarter"] == quarter), ["cik", "cusip", "weight"]]
    r = load_returns(ny, nq)
    m = h.merge(r, on="cusip", how="inner")
    m["contrib"] = m["weight"] * m["log_return"]
    return m.groupby("cik")["contrib"].sum().rename("profitability")


def zscore(df, cols):
    out = df.copy()
    for c in cols:
        v = out[c].astype(float).replace([np.inf, -np.inf], np.nan)
        v = v.fillna(v.median() if v.notna().any() else 0.0)
        sd = v.std()
        out[c] = (v - v.mean()) / sd if sd > 0 else 0.0
    return out

In [ ]:
def build_feature_graph(year, quarter, col):
    edges = load_edges(year, quarter, col)
    if edges.empty:
        raise RuntimeError(f"no Δ-edges for {year} Q{quarter}")
    aum = load_aum(year, quarter)
    py, pq = prev_year_quarter(year, quarter)
    try:
        past_prof = investor_profitability(py, pq).reset_index()
    except Exception:
        past_prof = pd.DataFrame(columns=["cik", "profitability"])

    cik_nh = edges.groupby("cik").size().rename("n_holdings").reset_index()
    cik_df = aum.merge(cik_nh, on="cik", how="outer").merge(past_prof, on="cik", how="left")
    cik_df["aum"] = cik_df["aum"].fillna(
        cik_df["aum"].median() if cik_df["aum"].notna().any() else 0.0)
    cik_df["log_aum"] = np.log(cik_df["aum"].clip(lower=1.0))
    cik_df["n_holdings"] = cik_df["n_holdings"].fillna(0)
    cik_df["profitability"] = cik_df["profitability"].fillna(0.0)
    CIK_FEATS = ["log_aum", "n_holdings", "profitability"]
    cik_df = zscore(cik_df, CIK_FEATS)
    stock_df = zscore(load_stock_features(year, quarter), STOCK_FEATURE_COLS)

    cik_ids = pd.Index(edges["cik"].unique())
    cusip_ids = pd.Index(edges["cusip"].unique())
    cik_df = cik_df.set_index("cik").reindex(cik_ids).fillna(0.0)
    stock_df = stock_df.set_index("cusip").reindex(cusip_ids).fillna(0.0)
    F_cik = cik_df[CIK_FEATS].to_numpy()
    F_stk = stock_df[STOCK_FEATURE_COLS].to_numpy()
    Fdim = F_cik.shape[1] + F_stk.shape[1]
    X = np.zeros((len(cik_ids) + len(cusip_ids), Fdim), dtype=np.float32)
    X[:len(cik_ids), :F_cik.shape[1]] = F_cik
    X[len(cik_ids):, F_cik.shape[1]:] = F_stk
    cik_pos = {c: i for i, c in enumerate(cik_ids)}
    cusip_pos = {c: i + len(cik_ids) for i, c in enumerate(cusip_ids)}

    edges = edges.copy()
    edges["src"] = edges["cik"].map(cik_pos).astype(np.int64)
    edges["dst"] = edges["cusip"].map(cusip_pos).astype(np.int64)
    return {
        "X": X,
        "edges": edges[["src", "dst", "w"]].reset_index(drop=True),
        "n_cik": len(cik_ids),
        "n_cusip": len(cusip_ids),
        "n_nodes": len(cik_ids) + len(cusip_ids),
        "cik_ids": cik_ids,
        "cusip_ids": cusip_ids,
    }


def split_edges(edges_df, tr, va, te, seed):
    df = edges_df.sample(frac=1, random_state=seed).reset_index(drop=True)
    n = len(df); n_tr = int(n * tr); n_va = int(n * va)
    train = df.iloc[:n_tr].reset_index(drop=True)
    val   = df.iloc[n_tr:n_tr + n_va].reset_index(drop=True)
    test  = df.iloc[n_tr + n_va:].reset_index(drop=True)
    train_nodes = set(train["src"]).union(train["dst"])
    def reflow(part):
        m = ~part["src"].isin(train_nodes) | ~part["dst"].isin(train_nodes)
        return part[~m].reset_index(drop=True), part[m]
    val,  vm = reflow(val)
    test, tm = reflow(test)
    if len(vm) + len(tm) > 0:
        train = pd.concat([train, vm, tm], ignore_index=True)
    return train, val, test


def edges_to_index(df, device):
    src = df["src"].to_numpy(); dst = df["dst"].to_numpy()
    w = df["w"].to_numpy().astype(np.float32)
    ei = np.stack([np.concatenate([src, dst]), np.concatenate([dst, src])], axis=0)
    ew = np.concatenate([np.abs(w), np.abs(w)])
    return (torch.tensor(ei, dtype=torch.long, device=device),
            torch.tensor(ew, dtype=torch.float, device=device))


def sample_bipartite_negatives(pos_edges_df, n_cik, n_cusip,
                               num_negatives, seed, forbid_edges_df=None):
    if forbid_edges_df is None:
        forbid_edges_df = pos_edges_df
    forbid = set(zip(forbid_edges_df["src"].astype(int).tolist(),
                     forbid_edges_df["dst"].astype(int).tolist()))
    rng = np.random.default_rng(seed)
    target = len(pos_edges_df) * num_negatives
    out = []
    while len(out) < target:
        u = rng.integers(0, n_cik, size=max(target * 2, 10000))
        v = rng.integers(n_cik, n_cik + n_cusip, size=max(target * 2, 10000))
        for a, b in zip(u, v):
            if (int(a), int(b)) not in forbid:
                out.append((int(a), int(b)))
                if len(out) >= target: break
    return np.asarray(out, dtype=np.int64)

In [ ]:
class WeightedLightGCN(nn.Module):
    def __init__(self, in_feats, embedding_dim, num_layers):
        super().__init__()
        self.input_proj = nn.Linear(in_feats, embedding_dim)
        nn.init.normal_(self.input_proj.weight, std=0.1)
        nn.init.zeros_(self.input_proj.bias)
        self.convs = nn.ModuleList([
            GCNConv(embedding_dim, embedding_dim,
                    improved=False, cached=False, add_self_loops=True)
            for _ in range(num_layers)
        ])

    def forward(self, x, edge_index, edge_weight):
        x = self.input_proj(x)
        layers = [x]
        for conv in self.convs:
            x = conv(x, edge_index, edge_weight=edge_weight)
            layers.append(x)
        return torch.stack(layers, dim=0).mean(dim=0)


def bpr_loss(pos_scores, neg_scores):
    return -torch.log(torch.sigmoid(pos_scores - neg_scores) + 1e-10).mean()


def train_model(model, X, train_ei, train_ew, trainval_ei, trainval_ew,
                train_pos, train_neg, val_pos, val_neg,
                epochs, lr, patience, show_progress=False):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    train_losses, val_losses = [], []
    best_val = float("inf"); best_state = None; no_improve = 0
    iterator = tqdm(range(epochs), desc="train", unit="ep", leave=False) if show_progress else range(epochs)
    for epoch in iterator:
        model.train(); opt.zero_grad()
        emb = model(X, train_ei, train_ew)
        ps = (emb[train_pos[:,0]] * emb[train_pos[:,1]]).sum(dim=1)
        ns = (emb[train_neg[:,0]] * emb[train_neg[:,1]]).sum(dim=1)
        loss = bpr_loss(ps, ns); loss.backward(); opt.step()
        train_losses.append(loss.item())

        model.eval()
        with torch.no_grad():
            emb = model(X, trainval_ei, trainval_ew)
            vps = (emb[val_pos[:,0]] * emb[val_pos[:,1]]).sum(dim=1)
            vns = (emb[val_neg[:,0]] * emb[val_neg[:,1]]).sum(dim=1)
            v_loss = bpr_loss(vps, vns).item()
        val_losses.append(v_loss)

        if v_loss < best_val:
            best_val = v_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
        if no_improve >= patience:
            break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, train_losses, val_losses

## 5. Evaluation + per-stock ranking helpers

In [ ]:
def evaluate_test(model, X, edge_index, edge_weight, pos_pairs, neg_pairs,
                  threshold=0.5):
    model.eval()
    with torch.no_grad():
        emb = model(X, edge_index, edge_weight)
        pu = emb[pos_pairs[:,0]]; pv = emb[pos_pairs[:,1]]
        nu = emb[neg_pairs[:,0]]; nv = emb[neg_pairs[:,1]]
        ps = (pu * pv).sum(dim=1).sigmoid().cpu().numpy()
        ns = (nu * nv).sum(dim=1).sigmoid().cpu().numpy()
    scores = np.concatenate([ps, ns])
    labels = np.concatenate([np.ones(len(ps)), np.zeros(len(ns))])
    auc = roc_auc_score(labels, scores)
    pred = (scores > threshold).astype(int)
    return {
        "auc": auc,
        "precision": precision_score(labels, pred, zero_division=0),
        "recall":    recall_score(labels, pred, zero_division=0),
        "f1":        f1_score(labels, pred, zero_division=0),
        "scores":    scores, "labels": labels,
    }


def find_optimal_threshold(scores, labels):
    p, r, ths = precision_recall_curve(labels, scores)
    f1 = 2 * p * r / (p + r + 1e-10)
    idx = int(np.argmax(f1))
    th = ths[idx] if idx < len(ths) else 0.5
    return float(th), float(average_precision_score(labels, scores))


def evaluate_top_k(model, X, edge_index, edge_weight, test_edges_df,
                   n_cik, n_cusip, k_list):
    model.eval()
    with torch.no_grad():
        emb = model(X, edge_index, edge_weight)
    fund_to_stocks = {}
    for src, dst in zip(test_edges_df["src"].to_numpy(),
                        test_edges_df["dst"].to_numpy()):
        fund_to_stocks.setdefault(int(src), set()).add(int(dst))
    stock_indices = np.arange(n_cik, n_cik + n_cusip)
    stock_emb = emb[stock_indices]
    res = {k: {"hit_rate": 0.0, "ndcg": 0.0} for k in k_list}
    n_funds = len(fund_to_stocks)
    for fund_idx, true_stocks in fund_to_stocks.items():
        scores = (emb[fund_idx] * stock_emb).sum(dim=1).cpu().numpy()
        order = np.argsort(-scores)
        for k in k_list:
            top_k_global = stock_indices[order[:k]]
            hits = [int(s) for s in top_k_global if int(s) in true_stocks]
            if hits:
                res[k]["hit_rate"] += 1
                dcg = sum(
                    1.0 / np.log2(i + 2)
                    for i, s in enumerate(top_k_global) if int(s) in true_stocks)
                idcg = sum(1.0 / np.log2(i + 2) for i in range(min(len(true_stocks), k)))
                res[k]["ndcg"] += dcg / idcg if idcg > 0 else 0.0
    for k in k_list:
        res[k]["hit_rate"] /= max(n_funds, 1)
        res[k]["ndcg"]     /= max(n_funds, 1)
    return res


def compute_stock_ranking(model, X, edge_index, edge_weight,
                          n_cik, n_cusip, cusip_ids, year, quarter,
                          score_chunk_funds=2048):
    """Per-stock score = mean over all funds of sigmoid(fund_emb · stock_emb).
    Returns df sorted descending by score with rank=1..n_stocks."""
    model.eval()
    with torch.no_grad():
        emb = model(X, edge_index, edge_weight)
        fund_emb  = emb[:n_cik]                          # (F, D)
        stock_emb = emb[n_cik:n_cik + n_cusip]           # (S, D)
        # Chunk over funds to control peak memory: (chunk, D) @ (D, S)
        accum = torch.zeros(n_cusip, device=emb.device)
        for start in range(0, n_cik, score_chunk_funds):
            end = min(start + score_chunk_funds, n_cik)
            block = torch.sigmoid(fund_emb[start:end] @ stock_emb.T)  # (chunk, S)
            accum += block.sum(dim=0)
        mean_score = (accum / max(n_cik, 1)).cpu().numpy()

    df = pd.DataFrame({
        "cusip":   list(cusip_ids),
        "year":    int(year),
        "quarter": int(quarter),
        "mean_score": mean_score,
    }).sort_values("mean_score", ascending=False).reset_index(drop=True)
    df["rank"] = np.arange(1, len(df) + 1, dtype=np.int64)
    return df

## 6. Per-quarter runner

In [ ]:
def run_quarter(year, quarter, edges_col):
    """Train + evaluate one quarter. Returns (metrics_dict, ranking_df)."""
    t0 = time.perf_counter()
    G = build_feature_graph(year, quarter, edges_col)
    train_e, val_e, test_e = split_edges(
        G["edges"], TRAIN_RATIO, VAL_RATIO, TEST_RATIO, RANDOM_SEED)

    train_ei, train_ew = edges_to_index(train_e, DEVICE)
    trainval_df = pd.concat([train_e, val_e], ignore_index=True)
    tv_ei, tv_ew = edges_to_index(trainval_df, DEVICE)
    X_t = torch.tensor(G["X"], dtype=torch.float, device=DEVICE)

    train_pos = torch.tensor(train_e[["src","dst"]].to_numpy(np.int64), device=DEVICE)
    train_neg_np = sample_bipartite_negatives(
        train_e, G["n_cik"], G["n_cusip"], 1, RANDOM_SEED)
    train_neg = torch.tensor(train_neg_np, device=DEVICE)
    val_pos = torch.tensor(val_e[["src","dst"]].to_numpy(np.int64), device=DEVICE)
    val_neg_np = sample_bipartite_negatives(
        val_e, G["n_cik"], G["n_cusip"], 1, RANDOM_SEED + 1, trainval_df)
    val_neg = torch.tensor(val_neg_np, device=DEVICE)

    model = WeightedLightGCN(G["X"].shape[1], EMBEDDING_DIM, NUM_LAYERS).to(DEVICE)
    model, train_losses, val_losses = train_model(
        model, X_t, train_ei, train_ew, tv_ei, tv_ew,
        train_pos, train_neg, val_pos, val_neg,
        EPOCHS, LEARNING_RATE, EARLY_STOPPING_PATIENCE, show_progress=False)

    test_pos = torch.tensor(test_e[["src","dst"]].to_numpy(np.int64), device=DEVICE)
    forbid_all = pd.concat([trainval_df, test_e], ignore_index=True)
    test_neg_np = sample_bipartite_negatives(
        test_e, G["n_cik"], G["n_cusip"], 1, RANDOM_SEED + 2, forbid_all)
    test_neg = torch.tensor(test_neg_np, device=DEVICE)

    res = evaluate_test(model, X_t, tv_ei, tv_ew, test_pos, test_neg, threshold=0.5)
    opt_th, ap = find_optimal_threshold(res["scores"], res["labels"])
    res_opt = evaluate_test(model, X_t, tv_ei, tv_ew, test_pos, test_neg, threshold=opt_th)
    top_k = evaluate_top_k(model, X_t, tv_ei, tv_ew, test_e,
                            G["n_cik"], G["n_cusip"], TOP_K_VALUES)

    rank_df = compute_stock_ranking(model, X_t, tv_ei, tv_ew,
                                     G["n_cik"], G["n_cusip"],
                                     G["cusip_ids"], year, quarter)

    metrics = {
        "year": int(year), "quarter": int(quarter),
        "n_funds": int(G["n_cik"]), "n_stocks": int(G["n_cusip"]),
        "n_edges": int(len(G["edges"])),
        "n_train": int(len(train_e)), "n_val": int(len(val_e)),
        "n_test":  int(len(test_e)),
        "epochs_trained": int(len(train_losses)),
        "final_train_loss": float(train_losses[-1]),
        "final_val_loss":   float(val_losses[-1]),
        "best_val_loss":    float(min(val_losses)),
        "auc":           float(res["auc"]),
        "avg_precision": float(ap),
        "opt_threshold": float(opt_th),
        "precision_05":  float(res["precision"]),
        "recall_05":     float(res["recall"]),
        "f1_05":         float(res["f1"]),
        "precision_opt": float(res_opt["precision"]),
        "recall_opt":    float(res_opt["recall"]),
        "f1_opt":        float(res_opt["f1"]),
        **{f"hit{k}":  float(top_k[k]["hit_rate"]) for k in TOP_K_VALUES},
        **{f"ndcg{k}": float(top_k[k]["ndcg"])     for k in TOP_K_VALUES},
        "elapsed_s": float(time.perf_counter() - t0),
    }

    # Free GPU
    del model, X_t, train_ei, train_ew, tv_ei, tv_ew
    del train_pos, train_neg, val_pos, val_neg, test_pos, test_neg
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return metrics, rank_df

## 7. Resumable I/O helpers

In [ ]:
RESULTS_CSV = RESULTS_PATH / f"sweep_results__{EDGES_COLUMN}.csv"
RANKS_PARQUET = RESULTS_PATH / f"cusip_ranks__{EDGES_COLUMN}.parquet"
print(f"results CSV    : {RESULTS_CSV}")
print(f"ranks parquet  : {RANKS_PARQUET}")

def list_available_quarters(edges_col):
    df = CHANGED_HOLDINGS
    sub = df.loc[df[edges_col].notna(), ["year", "quarter"]].drop_duplicates()
    yq = sorted({(int(y), int(q)) for y, q in sub.itertuples(index=False)})
    avail = set(yq)
    # need previous quarter for the `profitability` feature -- skip very first
    return [(y, q) for (y, q) in yq if prev_year_quarter(y, q) in avail]


def load_done_set(csv_path):
    if not csv_path.exists():
        return set()
    df = pd.read_csv(csv_path)
    return set(zip(df["year"].astype(int), df["quarter"].astype(int)))


def append_metrics_row(row, csv_path):
    pd.DataFrame([row]).to_csv(
        csv_path, mode="a", header=not csv_path.exists(), index=False)


def append_ranks(rank_df, parquet_path, model_tag="WeightedLightGCN"):
    rank_df = rank_df.copy()
    rank_df["model"] = model_tag
    if parquet_path.exists():
        prev = pd.read_parquet(parquet_path)
        y = int(rank_df["year"].iloc[0]); q = int(rank_df["quarter"].iloc[0])
        prev = prev[~((prev["year"] == y) & (prev["quarter"] == q) & (prev["model"] == model_tag))]
        rank_df = pd.concat([prev, rank_df], ignore_index=True)
    rank_df.to_parquet(parquet_path, index=False)

## 8. Sweep

Trains one quarter at a time, appends to CSV + parquet after each one. Re-running this cell will resume from where it left off (skips quarters already in `RESULTS_CSV`). To force a full rerun, delete the CSV first.

In [ ]:
all_quarters = list_available_quarters(EDGES_COLUMN)
if QUARTERS_OVERRIDE is not None:
    requested = [(int(y), int(q)) for y, q in QUARTERS_OVERRIDE]
    quarters = [yq for yq in requested if yq in set(all_quarters)]
    skipped = [yq for yq in requested if yq not in set(all_quarters)]
    if skipped:
        print(f"  [skip] not in available quarters: {skipped}")
else:
    quarters = all_quarters

done = load_done_set(RESULTS_CSV)
remaining = [yq for yq in quarters if yq not in done]
print(f"available quarters: {len(all_quarters)}  selected: {len(quarters)}  done: {len(done)}  remaining: {len(remaining)}")
if remaining:
    print(f"  range: {remaining[0]} .. {remaining[-1]}")

t_start = time.perf_counter()
failures = []
for i, (y, q) in enumerate(remaining, 1):
    try:
        metrics, rank_df = run_quarter(y, q, EDGES_COLUMN)
        append_metrics_row(metrics, RESULTS_CSV)
        append_ranks(rank_df, RANKS_PARQUET)
        elapsed = time.perf_counter() - t_start
        eta = elapsed / i * max(len(remaining) - i, 0)
        print(f"[{len(done)+i:2d}/{len(quarters)}] {y} Q{q}  "
              f"AUC={metrics['auc']:.4f}  hit@10={metrics['hit10']:.4f}  "
              f"ndcg@10={metrics['ndcg10']:.4f}  "
              f"({metrics['epochs_trained']:>2}ep, {metrics['elapsed_s']:.0f}s)  "
              f"ETA {eta/60:.1f}m")
    except Exception as e:
        failures.append((y, q, repr(e)))
        print(f"  ! {y} Q{q} FAILED: {e.__class__.__name__}: {e}")
        traceback.print_exc()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

total_min = (time.perf_counter() - t_start) / 60
print(f"\nsweep complete in {total_min:.1f} min  ({len(remaining) - len(failures)} ok, {len(failures)} failed)")
if failures:
    print("failed quarters:")
    for f in failures:
        print(f"  {f}")

## 9. Inspect results

In [ ]:
if RESULTS_CSV.exists():
    res_df = pd.read_csv(RESULTS_CSV).sort_values(["year", "quarter"]).reset_index(drop=True)
    print(f"quarters in results: {len(res_df)}")
    cols = ["year", "quarter", "n_funds", "n_stocks", "n_edges",
            "auc", "avg_precision", "f1_05", "hit10", "ndcg10", "epochs_trained"]
    print(res_df[cols].to_string(index=False))
    print()
    print("Aggregates (mean ± std):")
    agg_cols = ["auc", "avg_precision", "f1_05", "hit5", "hit10", "hit20", "ndcg10"]
    print(res_df[agg_cols].agg(["mean", "std"]).T.round(4))
else:
    print("no results yet")

In [ ]:
# Plot AUC and hit@10 over time
if RESULTS_CSV.exists() and len(res_df) > 0:
    res_df["yq"] = res_df["year"] + (res_df["quarter"] - 1) / 4
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    axes[0].plot(res_df["yq"], res_df["auc"], marker="o", color="steelblue")
    axes[0].set_xlabel("Year"); axes[0].set_ylabel("AUC-ROC")
    axes[0].set_title(f"WeightedLightGCN AUC across {len(res_df)} quarters")
    axes[0].grid(True); axes[0].axhline(0.5, color="gray", linestyle="--", alpha=0.5)

    axes[1].plot(res_df["yq"], res_df["hit10"],  marker="o", label="Hit@10")
    axes[1].plot(res_df["yq"], res_df["ndcg10"], marker="s", label="NDCG@10")
    axes[1].set_xlabel("Year"); axes[1].set_ylabel("score")
    axes[1].set_title("Top-10 ranking quality")
    axes[1].grid(True); axes[1].legend()
    plt.tight_layout(); plt.show()

In [ ]:
# Per-quarter top stocks (latest only)
if RANKS_PARQUET.exists():
    ranks = pd.read_parquet(RANKS_PARQUET)
    print(f"total ranking rows: {len(ranks):,}")
    print(f"quarters with ranks: {ranks[['year','quarter']].drop_duplicates().shape[0]}")
    print()
    last_y = ranks["year"].max()
    last_q = ranks.loc[ranks["year"] == last_y, "quarter"].max()
    latest = ranks[(ranks["year"] == last_y) & (ranks["quarter"] == last_q)]
    print(f"top 20 stocks for {last_y} Q{last_q}:")
    print(latest.head(20).to_string(index=False))
else:
    print("no ranks parquet yet")

## 10. Notes

- **Score definition**: `mean_score = mean over all funds of sigmoid(fund_emb · stock_emb)` — interpretable as average attractiveness of the stock to the fund universe.
- **Resumable**: this notebook reads `sweep_results__<col>.csv` and skips any (year, quarter) already present. Delete the CSV (and parquet, to avoid stale ranks) to force a full rerun.
- **Memory**: per-quarter peak is dominated by the bipartite affinity matrix `(n_funds × n_stocks)` computed during ranking. With ~5 K funds × ~3 K stocks × FP32 ≈ 60 MB, this is comfortable on the 6 GB laptop GPU. The `score_chunk_funds=2048` arg in `compute_stock_ranking` controls the chunk size if you ever need to rein it in.
- **Sign of Δw**: BPR sees edge existence (any sign) as positive; GCN propagation uses `|Δw|` as the message scale. To filter by sign, edit `load_edges` to add a post-condition like `df[df['w'] > 0]`.
- **Per-fund top-K rankings**: not saved (would balloon the parquet to ~ funds × K rows). If you want them, add a `compute_fund_recommendations` cell that mirrors `compute_stock_ranking` but iterates over funds.